In [25]:
from operator import add
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END, START
from pydantic import BaseModel, Field
from typing import Annotated, TypedDict, Literal

In [26]:
OLLAMA_MODEL = "qwen2.5-coder:14b"
OLLAMA_BASE_URL = "http://localhost:11434"
model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

In [27]:
class SentimentState(BaseModel):
    sentiment:Literal['positive','negative']=Field(description="Holds wether the review of customer was positive or negative")

class diagnosisState(BaseModel):
    issue_type:Literal['UI','Authentication','Ordering','Other'] =Field(description="Holds what type of issue was faced by the customere")
    tone:Literal['anger','frustration','Other'] =Field(description="Holds what emotion the customer is experiencing")
    severity:Literal['low','medium','high','critical'] =Field(description="Holds the severity of the problem")


In [28]:
structured_sentiment= model.with_structured_output(SentimentState)
structured_diagnosis= model.with_structured_output(diagnosisState)

In [29]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal['positive','negative']
    diagnosis:dict
    response:str

In [30]:
def classify_sentiment(state:ReviewState):
    template=PromptTemplate(
        input_variables=['review'],
        template="Classify the Given review into positive or negative. Review= {review}"
    )
    prompt=template.format(review=state['review'])

    response=structured_sentiment.invoke(prompt)

    return {'sentiment':response}

In [31]:
def run_diagnosis(state:ReviewState):
    template=PromptTemplate(
        input_variables=['review'],
        template="The customer has given the followng review {review}, you need to write the issue type, 'UI','Authentication','Ordering','Other',"
                 "tone: 'anger','frustration','Other'"
                 "severity: low','medium','high','critical'"

    )
    prompt=template.format(review=state['review'])

    response=structured_diagnosis.invoke(prompt)

    return {'diagnosis':response.model_dump()}

In [32]:
def positive_response(state:ReviewState):
    template=PromptTemplate(
        input_variables=['review'],
        template="The customer has given the followng positive review {review}, write a elegant response thanking them for the review and suggesting them to also review on the website"

    )
    prompt=template.format(review=state['review'])

    response=model.invoke(prompt).content

    return {'response':response}

In [33]:
def negative_response(state:ReviewState):
    template=PromptTemplate(
        input_variables=['review','diagnosis'],
        template="The customer has given the followng negative review {review} with the following diagnosis {diagnosis}, write a elegant response highlight their concern  "

    )
    prompt=template.format(review=state['review'],diagnosis=state['diagnosis'])

    response=model.invoke(prompt).content

    return {'response':response}

In [34]:
def check_sentiment(state:ReviewState)-> Literal['positive_response','run_diagnosis']:
    if state["sentiment"]== "positive":
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [35]:
graph=StateGraph(ReviewState)

graph.add_node('classify_sentiment',classify_sentiment)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('positive_response',positive_response)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'classify_sentiment')
graph.add_conditional_edges('classify_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)

workflow=graph.compile()


In [36]:
initial_state={
    'review':"The app was really bad couldnt even login"
}
workflow.invoke(initial_state)

{'review': 'The app was really bad couldnt even login',
 'sentiment': SentimentState(sentiment='negative'),
 'diagnosis': {'issue_type': 'Authentication',
  'tone': 'anger',
  'severity': 'low'},
 'response': "Dear Customer,\n\nThank you for bringing this to our attention. We sincerely apologize for any inconvenience you experienced while trying to log in to our app. It's important to us that all users have a smooth and secure experience, and we understand your frustration.\n\nWe've identified the issue as related to authentication and are working diligently to resolve it. Our team is reviewing the situation and will take appropriate measures to ensure this doesn't happen again in the future.\n\nPlease rest assured that we value your feedback and are committed to improving our service. If you need any assistance or have further concerns, please don't hesitate to reach out to us.\n\nThank you for your understanding and patience.\n\nBest regards,\n[Your Company Name]"}